# Part 12: Stress Composite Sensitivity

This notebook runs the reproducible Part 12 Python runner in Colab. Part 12 keeps the Part 1 HMM-4 state partition frozen and varies only the economic stress ordering used to relabel states.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os

PROJECT_DIR = Path('/content/drive/MyDrive/project_edhec_paper')
os.chdir(PROJECT_DIR)
print('Working directory:', Path.cwd())

assert (PROJECT_DIR / 'data_2026' / 'cleaned').exists(), 'Cleaned input folder not found.'
assert (PROJECT_DIR / 'experiments' / 'part12_stress_composite_sensitivity' / 'run_part12_stress_composite_sensitivity.py').exists(), 'Part 12 runner not found.'

In [ ]:
!python -m pip install -q -r experiments/part12_stress_composite_sensitivity/requirements-part12.txt

## Path configuration

Part 12 requires completed Part 1, Part 2, and Part 4 runs. The helper first tries the canonical Drive path, then the downloaded `*_outputs/...` folder structure.

In [ ]:
from pathlib import Path

def choose_existing(*candidates):
    for candidate in candidates:
        path = Path(candidate)
        if path.exists():
            return path
    return Path(candidates[0])

INPUT_DIR = Path('data_2026/cleaned')
PART1_RUN_DIR = choose_existing(
    'outputs/part1_btc_macro_state/colab_part1_seed42',
    'outputs/part1_btc_macro_state_outputs/part1_btc_macro_state/colab_part1_seed42',
)
PART2_RUN_DIR = choose_existing(
    'outputs/part2_portfolio_risk_budget/colab_part2_seed42',
    'outputs/part2_portfolio_risk_budget_outputs/part2_portfolio_risk_budget/colab_part2_seed42',
)
PART4_RUN_DIR = choose_existing(
    'outputs/part4_conditional_btc_allocation/colab_part4_seed42',
    'outputs/part4_conditional_btc_allocation_outputs/part4_conditional_btc_allocation/colab_part4_seed42',
)
OUTPUT_DIR = Path('outputs/part12_stress_composite_sensitivity_outputs/part12_stress_composite_sensitivity')
RUN_ID = 'colab_part12_seed42'

print('INPUT_DIR:', INPUT_DIR)
print('PART1_RUN_DIR:', PART1_RUN_DIR)
print('PART2_RUN_DIR:', PART2_RUN_DIR)
print('PART4_RUN_DIR:', PART4_RUN_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)
assert PART1_RUN_DIR.exists(), 'Part 1 run directory not found. Run Part 1 first.'
assert PART2_RUN_DIR.exists(), 'Part 2 run directory not found. Run Part 2 first.'
assert PART4_RUN_DIR.exists(), 'Part 4 run directory not found. Run Part 4 first.'

In [ ]:
!python experiments/part12_stress_composite_sensitivity/run_part12_stress_composite_sensitivity.py \
  --input-dir "{INPUT_DIR}" \
  --part1-run-dir "{PART1_RUN_DIR}" \
  --part2-run-dir "{PART2_RUN_DIR}" \
  --part4-run-dir "{PART4_RUN_DIR}" \
  --output-dir "{OUTPUT_DIR}" \
  --run-id "{RUN_ID}" \
  --seed 42

In [ ]:
import json
import pandas as pd

RUN_DIR = OUTPUT_DIR / RUN_ID
RESULTS = RUN_DIR / 'results'
FIGURES = RUN_DIR / 'figures'

validation = json.loads((RESULTS / 'output_validation_summary.json').read_text())
print(json.dumps(validation, indent=2))
assert validation['status'] == 'passed'

In [ ]:
display(pd.read_csv(RESULTS / 'part12_label_stability_vs_original.csv'))
display(pd.read_csv(RESULTS / 'part12_state_relabeling_summary.csv'))
display(pd.read_csv(RESULTS / 'part12_rule_weight_by_sorting_method.csv'))
display(pd.read_csv(RESULTS / 'part12_rule_sensitivity_summary.csv'))

In [ ]:
findings = json.loads((RESULTS / 'part12_key_findings.json').read_text())
print(json.dumps(findings, indent=2))

In [ ]:
from IPython.display import Image, display

for name in [
    'part12_state_relabeling_heatmap.png',
    'part12_rule_sensitivity_by_sorting_method.png',
]:
    print(name)
    display(Image(filename=str(FIGURES / name)))

In [ ]:
import shutil

zip_path = shutil.make_archive(str(RUN_DIR), 'zip', root_dir=RUN_DIR)
print('Created zip:', zip_path)